# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant metadata and print dataset description
dataset = mlc.Dataset(url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}, Version: {meta.version}")
print(f"Identifier: {meta.identifier}")
print(f"Keywords: {', '.join(meta.keywords)}")
print(f"License: {meta.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs. The dataset may define multiple record sets, each with distinct fields (columns).

**All entities are referenced by their `@id` field.**

List record sets and fields by their `@id`:

In [ ]:
# List all record sets with their @id and their fields

# Get top-level metadata dict
metadata_dict = meta.to_json()

record_sets = metadata_dict.get('recordSet', [])

if isinstance(record_sets, dict):
    record_sets = [record_sets]  # Handle single recordSet

if not record_sets:
    # Try schema.org convention
    record_sets = metadata_dict.get('record_sets', [])

if not record_sets:
    print('No record sets found in metadata.')
else:
    for record_set in record_sets:
        rec_id = record_set.get('@id', '<unknown>')
        print(f"RecordSet @id: {rec_id}")
        rec_fields = record_set.get('field', [])
        if isinstance(rec_fields, dict):
            rec_fields = [rec_fields]
        print("  Fields (@id):")
        for field in rec_fields:
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
            elif isinstance(field, str):
                print(f"    - {field}")
        print("")

# For this dataset, let's also print the recordSet @id list
rec_set_ids = []
for record_set in record_sets:
    rec_set_ids.append(record_set.get('@id'))
print(f"Record set @id's found: {rec_set_ids}")

# If no record set is listed in metadata, try to guess a default
if not rec_set_ids:
    # Attempt to infer from distribution
    distributions = metadata_dict.get('distribution', [])
    print(f"Distributions: {distributions}")
    # There may be only one record set for the dataset


## Display a sample record from a record set

Here, we enumerate the first few records from a record set by its `@id`. *If the dataset has only one record set, its `@id` is used below.*

In [ ]:
# Try to get first record set @id
if rec_set_ids:
    selected_record_set = rec_set_ids[0]
else:
    # This dataset may keep a single implicit record set, typically as 'dataset' or 'main' or via distribution
    # Try a default name
    selected_record_set = None

if selected_record_set:
    print(f"Sample records from {selected_record_set}:")
    for i, record in enumerate(dataset.records(record_set=selected_record_set)):
        print(record)
        if i >= 2:
            break
else:
    print('No explicit record set @id found; attempting to load all records (if available).')
    # Try with default
    for i, record in enumerate(dataset.records()):
        print(record)
        if i >= 2:
            break


## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for further analysis.

**Use the record set and field `@id`s from the overview above.**

If only one record set is present, use that.

In [ ]:
# Extract data from each record set by their @id

if not rec_set_ids:
    # Fallback: use None (default record set)
    rec_set_ids = [None]

dataframes = {}
for rec_id in rec_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        col_list = df.columns.tolist() if not df.empty else []
        print(f"Loaded {len(df)} records for record set {rec_id}. Fields: {col_list}")
    except Exception as e:
        print(f"Failed to load record set {rec_id}: {e}")

# Set main_record_set for further processing
main_record_set = rec_set_ids[0]
df = dataframes[main_record_set]
print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, and group data. 

We'll:
- Select a numeric field (e.g., peptide counts, coverage, MW)
- Filter out records below a threshold
- Normalize the field
- Group by a categorical field, if available


In [ ]:
# Choose a numeric field based on data columns
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or df[col].dtype == object]

# Attempt to auto-select typical mass spec columns
auto_numeric = None
for candidate in ['coverage', 'Coverage', 'MW', 'Abundance', 'peptide_counts', 'PeptideCounts', 'peptide_count', 'PSMs']:
    if candidate in df.columns:
        auto_numeric = candidate
        break
if not auto_numeric and numeric_candidates:
    # Try to infer numeric by conversion
    for col in numeric_candidates:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].notnull().sum() > 0:
                auto_numeric = col
                break
        except Exception:
            continue

numeric_field = auto_numeric if auto_numeric else None

if numeric_field:
    print(f"Selected numeric field: {numeric_field}")

    # Drop missing values
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = np.nanpercentile(df[numeric_field], 75)  # Filter top 25% as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field}:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find candidate group-by columns
    group_candidates = [col for col in df.columns if col.lower() in ['sample', 'condition', 'experiment', 'group', 'treatment', 'category']]
    group_field = group_candidates[0] if group_candidates else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name='mean_' + numeric_field)
        print(f"Grouped by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No group field found for grouping analysis.")
else:
    print("No numeric field found in the record set. Please inspect dataframe columns: ", df.columns.tolist())


## 5. Visualization
Visualize numeric data distribution (histogram) and relationship between two fields (scatterplot), using the EDA columns identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram of numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # Pairwise scatter if there are at least two numeric fields
    numeric_cols = []
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notnull().sum() > 0:
                numeric_cols.append(col)
        except:
            continue
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(
            data=df,
            x=numeric_cols[0],
            y=numeric_cols[1],
            alpha=0.6
        )
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"{numeric_cols[0]} vs {numeric_cols[1]}")
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field available for visualization.')


## 6. Conclusion
In this notebook, we loaded and explored the FAIR\(^2\) mass spectrometry dataset using the `mlcroissant` library.

- We read metadata, reviewed record sets and fields (all referenced by `@id`).
- Data was loaded into Pandas DataFrames using Croissant methods.
- We performed EDA by filtering and normalizing a numeric field such as peptide counts, coverage, or abundance, and presented simple summary statistics.
- Visualizations illustrated the field distribution and highlighted pairwise feature relationships.

For further analyses, refer to the Croissant specification or [`mlcroissant` documentation](https://mlcommons.org/croissant/). Explore the [SenScience repository](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for additional data context and schema details.